# Dignity — train the risk head

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/crichalchemist/Dignity/blob/main/notebooks/train_risk.ipynb)

This notebook trains one Dignity model end to end on the synthetic transaction data. It is the
same path `dignity-train` runs, with the intermediate pieces visible:

1. load a YAML config and pick the device
2. `prepare_data` generates the data, runs the privacy stage once and splits by block 70/15/15
3. `fit` builds `Dignity(task=...)`, trains it and keeps `best.pt` by val PR-AUC
4. plot the curves, score the validation windows, look at the attention weights
5. export the best checkpoint to ONNX

A GPU runtime is recommended: `config/train_risk.yaml` takes about 2.5 minutes per epoch on an
8-core CPU and well under a minute on a T4. On Colab the setup cell clones the repo and installs
only the ONNX packages Colab lacks (everything else is preinstalled) and mounts Google Drive so
checkpoints and results outlive the runtime (`MOUNT_DRIVE = False` skips that).

In [ ]:
import os
import subprocess
import sys
from importlib.util import find_spec
from pathlib import Path

REPO = "https://github.com/crichalchemist/Dignity"
IN_COLAB = "google.colab" in sys.modules
MOUNT_DRIVE = True  # Colab only: keep checkpoints and results in Google Drive


def find_root(start: Path) -> Path | None:
    for candidate in (start, *start.parents):
        if (candidate / "pyproject.toml").exists() and (candidate / "core").is_dir():
            return candidate
    return None


root = find_root(Path.cwd())
if root is None and IN_COLAB and Path("/content/Dignity/pyproject.toml").exists():
    # Restarting the runtime resets the working directory to /content but keeps
    # the clone from the earlier run; cloning again over it would fail.
    root = Path("/content/Dignity")
if root is None:
    subprocess.run(["git", "clone", "--depth", "1", REPO], check=True)
    root = Path.cwd() / "Dignity"
elif IN_COLAB and root == Path("/content/Dignity"):
    # A clone left over from an earlier run of this runtime: bring it up to date.
    # Python caches imported modules, so after a pull restart the runtime before
    # re-running the cells below, or the old code keeps running.
    subprocess.run(["git", "-C", str(root), "pull", "--ff-only"], check=True)
os.chdir(root)
sys.path.insert(0, str(root))
print("repo root:", root)

if IN_COLAB:
    # Colab already ships torch, numpy, pandas, scipy, scikit-learn and matplotlib.
    # Do not `pip install -r requirements.txt` here: its numpy<2 pin makes pip try
    # to downgrade numpy underneath Colab's preinstalled stack, which runs for many
    # minutes and looks like a hang. Install only what is actually missing.
    needed = {"onnx": "onnx", "onnxruntime": "onnxruntime", "yaml": "pyyaml"}
    missing = [pkg for mod, pkg in needed.items() if find_spec(mod) is None]
    if missing:
        print("installing:", " ".join(missing))
        subprocess.run(
            [sys.executable, "-m", "pip", "install", "-q", *missing], check=True
        )
    if MOUNT_DRIVE:
        from google.colab import drive

        drive.mount("/content/drive")

## Configuration

Everything comes from one YAML file, exactly as on the command line. The four knobs below are
the ones worth changing from a notebook:

- `CONFIG`: `config/train_risk.yaml` is the shipped risk setup with the privacy stage on;
  `config/colab.yaml` is a smaller model with shorter windows for quick runs.
- `EPOCHS` and `CHECKPOINT_DIR` override the YAML when set. On Colab the setup cell mounts
  Google Drive and checkpoints default to `MyDrive/dignity/runs/<config>` (with a
  `_noprivacy` suffix when `PRIVACY = False`), so a run survives its runtime.
- `PRIVACY = False` drops the `privacy:` block. Run once with it on and once off to measure what
  the noise costs the model; the shipped noise scales are large on purpose (see
  `docs/PRIVACY.md`).

In [ ]:
import torch

from core.config import DignityConfig

CONFIG = "config/train_risk.yaml"
EPOCHS = None  # None keeps the YAML value
CHECKPOINT_DIR = None  # None: Google Drive when mounted (Colab), else the YAML value
PRIVACY = True  # False trains a no-privacy baseline for comparison

config = DignityConfig.from_yaml(CONFIG)
if EPOCHS is not None:
    config.train.epochs = EPOCHS
if CHECKPOINT_DIR is None and Path("/content/drive/MyDrive").is_dir():
    # Runtime storage disappears with the runtime: keep every run's checkpoints,
    # history and metrics in Drive, one folder per config and privacy setting.
    CHECKPOINT_DIR = "/content/drive/MyDrive/dignity/runs/" + Path(CONFIG).stem
    if not PRIVACY:
        CHECKPOINT_DIR += "_noprivacy"
if CHECKPOINT_DIR is not None:
    config.train.checkpoint_dir = CHECKPOINT_DIR
if not PRIVACY:
    config.privacy = None

device = torch.device(config.device if torch.cuda.is_available() else "cpu")
torch.manual_seed(config.seed)

print(f"task={config.model.task}  epochs={config.train.epochs}  device={device}")
print("checkpoints:", config.train.checkpoint_dir)
if config.privacy is None:
    print("privacy: off")
else:
    print(
        f"privacy: ε_total={config.privacy.epsilon_total}, "
        f"k={config.privacy.k}, columns={list(config.privacy.features)}"
    )

## Data

`prepare_data` is the same call `dignity-train` makes. `SyntheticGenerator` writes an event record: one row per step across `n_blocks` independent blocks, each anomalous with probability `contamination`. `TransactionPipeline.process_blocks` treats every block as one independent sequence: the privacy stage runs once on the whole frame, blocks are shuffled and split 70/15/15 at the block level into train, validation and test, the scaler is fit on the training blocks only, and signals and sliding windows are computed inside each block so nothing crosses a joint. This notebook trains on the train windows and validates on the validation windows; the test windows stay untouched.

In [ ]:
from train.loop import fit, prepare_data

df, data = prepare_data(config)

print("features used:", data.feature_names)
print(
    f"train windows {len(data.train.X)}, val windows {len(data.val.X)}, "
    f"test windows {len(data.test.X)}"
)

## Training

`fit` builds the model from the pipeline's feature width, trains one line per epoch and keeps `best.pt` by validation PR-AUC (`train.select_metric`) in the checkpoint directory, with `last.pt` and `history.json` beside it. The CLI calls the same function.

In [ ]:
result = fit(config, data, seed=config.seed, run_dir=config.train.checkpoint_dir)
print(
    f"best {config.train.select_metric} {result.best_metric:.4f} -> {result.best_path}"
)

## Curves

Training loss, and the validation PR-AUC and AUROC that pick `best.pt`.

In [ ]:
import matplotlib.pyplot as plt

epochs = [row["epoch"] for row in result.history]
fig, (ax_loss, ax_val) = plt.subplots(1, 2, figsize=(11, 3.5))
ax_loss.plot(epochs, [row["train_loss"] for row in result.history])
ax_loss.set(xlabel="epoch", ylabel="loss", title="train loss")
ax_val.plot(epochs, [row["val_pr_auc"] for row in result.history], label="PR-AUC")
ax_val.plot(epochs, [row["val_auroc"] for row in result.history], label="AUROC")
ax_val.set(xlabel="epoch", title="validation", ylim=(0, 1))
ax_val.legend()
plt.tight_layout()
plt.show()

## Validation scores

Reload `best.pt` and score every validation window. The validation loader is not shuffled, so the scores line up with `data.val.y_risk`. `risk_metrics` reports each metric with a block-bootstrap confidence interval.

In [ ]:
from data.loader import create_dataloader
from eval.metrics import risk_metrics
from train.checkpoint import build_model_from_meta, load_checkpoint
from train.engine import predict

ckpt = load_checkpoint(result.best_path)
model = build_model_from_meta(ckpt["meta"])
model.load_state_dict(ckpt["model_state"])
model.to(device)
scores = predict(
    model,
    create_dataloader(data.val.X, batch_size=config.data.batch_size, shuffle=False),
    device,
)["score"]
metrics = risk_metrics(
    data.val.y_risk,
    scores,
    data.val.block_index,
    n_boot=200,
    seed=config.seed,
    anomaly_type=data.val.anomaly_type,
)
print(metrics)

## Where the model looked

The backbone returns additive-attention weights over the window along with the prediction. One normal and one anomalous validation window, with the scaled `volume` feature behind the attention profile.

In [ ]:
import numpy as np

y_val = data.val.y_risk
feature = "volume" if "volume" in data.feature_names else None
column = data.feature_names.index(feature) if feature else 0
examples = [
    ("normal", int(np.flatnonzero(y_val == 0)[0])),
    ("anomalous", int(np.flatnonzero(y_val == 1)[0])),
]

model.eval()
fig, axes = plt.subplots(1, 2, figsize=(11, 3.5))
for ax, (name, idx) in zip(axes, examples, strict=True):
    x = torch.as_tensor(data.val.X[idx : idx + 1], dtype=torch.float32, device=device)
    with torch.no_grad():
        prediction, attention = model(x)
    attention = attention[0].cpu().numpy()
    ax.fill_between(range(len(attention)), attention, color="tab:red", alpha=0.35)
    ax.set(xlabel="step", ylabel="attention weight")
    ax.set_title(f"{name} window — score {float(prediction.reshape(-1)[0]):.2f}")
    twin = ax.twinx()
    twin.plot(data.val.X[idx, :, column], color="tab:gray", linewidth=1)
    twin.set_ylabel(f"{data.feature_names[column]} (scaled)")
plt.tight_layout()
plt.show()

## Export to ONNX

Same export the CLI does (`python -m export.to_onnx`). The exporter runs the model on CPU and
verifies the ONNX graph against PyTorch before returning.

In [ ]:
EXPORT = True

if EXPORT:
    from export.to_onnx import main as export_main

    export_main(
        [
            "--checkpoint",
            result.best_path,
            "--output",
            str(Path(config.train.checkpoint_dir) / "dignity_risk.onnx"),
        ]
    )

## Notes

- Everything here is synthetic data from `data/source/synthetic.py`; the numbers say the
  pipeline and model work, not that they detect anything real.
- To measure the privacy cost, run the notebook twice with `PRIVACY` on and off and compare
  the ROC AUC. The shipped Laplace scales (2000 for `volume`, 1000 for `price`) exercise the
  stage rather than tune it; `docs/PRIVACY.md` explains what each mechanism does and does not
  guarantee.
- The equivalent command line is `dignity-train --config <CONFIG>`; it writes the same
  checkpoints to `train.checkpoint_dir`.